# Somaiya Vidyavihar University
## K J Somaiya School of Engineering
### Department of Information Technology | Sem-VII | DiM (2026-27)
---
**Experiment No.:** 08  
**Title:** AI-Based Customer Persona Generation from Marketing Data  
**Batch:** A2 | **Roll No.:** 16010423075  
**Aim:** AI-Based Customer Persona Generation from Marketing Data.

### 1. Objective and Theoretical Overview
Customer Persona Generation creates data-driven, fictional profiles representing target consumer clusters.
In this experiment, unsupervised machine learning (**K-Means Clustering**) is applied to multi-attribute marketing data:
- **Demographic**: Age, Gender, Annual Income
- **Behavioral**: Spending Score (1-100), Purchase Frequency, Average Order Value
- **Channel Preference**: Social media, search, email, direct messaging

We determine optimal clusters via the **Elbow Method** and **Silhouette Coefficient**, normalize features using **StandardScaler**, train K-Means, visualize segment boundaries, and map mathematical centroids into actionable **Customer Personas** with digital marketing strategies.

In [1]:
# Step 1: Import Essential Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Visual settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
print('All required libraries imported successfully!')

In [2]:
# Step 2: Load the Customer Marketing Dataset
df = pd.read_csv('customer_marketing_data.csv')
print(f'Total records loaded: {len(df)}')
df.head()

In [3]:
# Step 3: Exploratory Data Analysis (EDA) & Data Cleaning
print('Missing Values Check:')
print(df.isnull().sum())

print('\nDuplicate Records:', df.duplicated().sum())

print('\nSummary Statistics:')
display(df.describe().T[['mean', 'std', 'min', '50%', 'max']])

In [4]:
# Step 4: Feature Selection & StandardScaler Normalization
features = ['Age', 'Annual_Income_k', 'Spending_Score', 'Purchase_Frequency', 'Avg_Order_Value']
X = df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Scaled Feature Matrix Shape:', X_scaled.shape)
print('Feature Means (Standardized):', np.round(X_scaled.mean(axis=0), 3))
print('Feature Stds (Standardized):', np.round(X_scaled.std(axis=0), 3))

In [5]:
# Step 5: Optimal Cluster Determination (Elbow Method & Silhouette Score)
k_range = range(2, 11)
wcss = [KMeans(n_clusters=1, random_state=42, n_init=10).fit(X_scaled).inertia_]
silhouette_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)
    score = silhouette_score(X_scaled, km.labels_)
    silhouette_scores.append(score)
    print(f'K = {k:2d} | WCSS: {km.inertia_:9.2f} | Silhouette Score: {score:.4f}')

# Visualization of Elbow & Silhouette Score
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(range(1, 11), wcss, marker='o', color='#2563eb', linewidth=2)
ax1.axvline(x=5, color='#ef4444', linestyle='--', label='Elbow Point (K=5)')
ax1.set_title('Elbow Method For Optimal K', fontweight='bold')
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Inertia (WCSS)')
ax1.legend()

ax2.plot(k_range, silhouette_scores, marker='s', color='#10b981', linewidth=2)
ax2.axvline(x=5, color='#ef4444', linestyle='--', label='Optimal K=5 (Score: 0.464)')
ax2.set_title('Silhouette Analysis For Cluster Separation', fontweight='bold')
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.legend()
plt.tight_layout()
plt.show()

In [6]:
# Step 6: Train Final K-Means Model (K=5)
optimal_k = 5
kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=15)
df['Cluster'] = kmeans_model.fit_predict(X_scaled)

print('Customer counts per cluster:')
print(df['Cluster'].value_counts().sort_index())

In [7]:
# Step 7: Cluster Centroid Analysis & Profiling
cluster_summary = df.groupby('Cluster')[features].mean()
print('Cluster Centroid Means (Original Units):')
display(cluster_summary.round(2))

In [8]:
# Step 8: Cluster Visualization (2D Scatter Plot & Centroids)
plt.figure(figsize=(10, 6))
palette = ['#ec4899', '#eab308', '#8b5cf6', '#06b6d4', '#10b981']

sns.scatterplot(
    data=df, 
    x='Annual_Income_k', 
    y='Spending_Score', 
    hue='Cluster', 
    palette=palette, 
    s=85, 
    alpha=0.85
)

# Overlay cluster centers
centroids = df.groupby('Cluster')[['Annual_Income_k', 'Spending_Score']].mean()
plt.scatter(centroids['Annual_Income_k'], centroids['Spending_Score'], s=250, c='black', marker='*', edgecolor='yellow', linewidth=1.5, label='Centroids')

plt.title('K-Means Customer Clusters (Annual Income vs Spending Score)', fontweight='bold')
plt.xlabel('Annual Income ($k)')
plt.ylabel('Spending Score (1-100)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [9]:
# Step 9: Map Clusters to Actionable Customer Personas
persona_names = {
    0: 'Trendsetting Impulse Buyer',
    1: 'Budget-Conscious Bargain Hunter',
    2: 'Elite VIP High-Roller',
    3: 'Cautious Affluent Saver',
    4: 'Mainstream Pragmatist'
}

df['Persona_Name'] = df['Cluster'].map(persona_names)

print('Persona Distribution:')
for p_id, name in persona_names.items():
    count = (df['Cluster'] == p_id).sum()
    pct = (count / len(df)) * 100
    print(f'Cluster {p_id} -> {name}: {count} customers ({pct:.1f}%)')

### Step 10: Tailored Digital Marketing Strategies by Persona

| Persona | Demographics | Behavioral Profile | Preferred Channel | Recommended Strategy |
| :--- | :--- | :--- | :--- | :--- |
| **Trendsetting Impulse Buyer** | 18–30 yrs, Entry Income | High Spend, High Frequency | Instagram Reels, TikTok, Push Notifications | Flash sales, FOMO urgency messaging, influencer collaboration, quick 10% abandoned cart discount codes. |
| **Budget-Conscious Bargain Hunter** | 35–65 yrs, Modest Income | Low Spend, Low Frequency | Email Newsletters, Google Search (Discounts) | BOGO coupons, price match guarantees, value bundling, free shipping thresholds. |
| **Elite VIP High-Roller** | 28–48 yrs, Top Earner ($106k+) | High Spend, High Order Value | VIP Email, Curated Instagram Showcase | Invitation-only pre-launches, white-glove concierge customer service, luxury loyalty tier incentives. |
| **Cautious Affluent Saver** | 38–68 yrs, High Earner ($104k+) | Low Spend, High Scrutiny | Google Search, LinkedIn, Technical Blogs | In-depth case studies, 10-year durability warranties, ROI calculations, transparent quality certifications. |
| **Mainstream Pragmatist** | 25–55 yrs, Middle Income | Moderate Spend, Steady Frequency | Google Search, Facebook, User Reviews | Social proof with 5-star review highlights, seasonal family bundles, loyalty points program. |